# Prepare Features for Databricks App

This notebook creates aggregated feature tables and syncs them to Lakebase for serving to Databricks Apps.
The synced tables provide low-latency OLTP access to aggregated features for the dashboard.

**Inputs:**
- Delta tables: articles, customers, transactions, article_features, customer_features

**Outputs:**
- Delta tables (Gold layer):
  - product_sales_summary
  - customer_demographics
  - time_series_sales

- Synced tables (Lakebase OLTP):
  - product_sales_summary_synced
  - customer_demographics_synced
  - time_series_sales_synced

**Lakebase Instance:** shared-online-store


## Setup


In [ ]:
import sys
sys.path.append("..")

from pyspark.sql.functions import *
from pyspark.sql.window import Window
import mlflow

from config.paths import MLFLOW_EXPERIMENT_DATA
from config.catalog_config import *

In [ ]:
mlflow.set_experiment(MLFLOW_EXPERIMENT_DATA)

## Load Base Tables


In [ ]:
# Load base tables
articles_df = spark.table(ARTICLES_TABLE)
customers_df = spark.table(CUSTOMERS_TABLE)
transactions_df = spark.table(TRANSACTIONS_TABLE)

print(f"Articles: {articles_df.count():,}")
print(f"Customers: {customers_df.count():,}")
print(f"Transactions: {transactions_df.count():,}")


## 1. Product Sales Summary Table

Top products with images and sales metrics for "Most Sold Products" dashboard.


In [ ]:
# Aggregate product sales
product_sales = (
    transactions_df
    .groupBy("article_id")
    .agg(
        count("*").alias("num_transactions"),
        sum("price").alias("total_revenue"),
        countDistinct("customer_id").alias("unique_customers"),
        avg("price").alias("avg_price"),
        max("t_dat").alias("last_purchase_date")
    )
)

# Join with article details
product_sales_summary = (
    product_sales
    .join(articles_df, on="article_id", how="left")
    .select(
        "article_id",
        "product_code",
        "prod_name",
        "product_type_name",
        "product_group_name",
        "colour_group_name",
        "department_name",
        "index_group_name",
        "section_name",
        "garment_group_name",
        "num_transactions",
        "total_revenue",
        "unique_customers",
        "avg_price",
        "last_purchase_date"
    )
)

print(f"Product sales summary: {product_sales_summary.count():,} products")
display(product_sales_summary.orderBy(col("num_transactions").desc()).limit(20))


In [ ]:
# Create synced table for product sales
product_sales_table = f"{CATALOG}.{SCHEMA}.product_sales_summary"

product_sales_summary.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(product_sales_table)

print(f"✓ Created table: {product_sales_table}")


## 2. Customer Demographics Table

Customer demographic aggregations for demographics dashboard.


In [ ]:
# Create age bins
customer_demo = (
    customers_df
    .withColumn(
        "age_bin",
        when(col("age") < 20, "<20")
        .when((col("age") >= 20) & (col("age") < 30), "20-29")
        .when((col("age") >= 30) & (col("age") < 40), "30-39")
        .when((col("age") >= 40) & (col("age") < 50), "40-49")
        .when((col("age") >= 50) & (col("age") < 60), "50-59")
        .when(col("age") >= 60, "60+")
        .otherwise("Unknown")
    )
)

# Join with transaction data for purchase statistics
customer_purchase_stats = (
    transactions_df
    .groupBy("customer_id")
    .agg(
        count("*").alias("num_purchases"),
        sum("price").alias("total_spent"),
        countDistinct("article_id").alias("unique_items"),
        max("t_dat").alias("last_purchase_date"),
        min("t_dat").alias("first_purchase_date")
    )
)

customer_demographics = (
    customer_demo
    .join(customer_purchase_stats, on="customer_id", how="left")
    .select(
        "customer_id",
        "age",
        "age_bin",
        "club_member_status",
        "fashion_news_frequency",
        "Active",
        "num_purchases",
        "total_spent",
        "unique_items",
        "last_purchase_date",
        "first_purchase_date"
    )
    .fillna(0, subset=["num_purchases", "total_spent", "unique_items"])
)

print(f"Customer demographics: {customer_demographics.count():,} customers")
display(customer_demographics.limit(20))


In [ ]:
# Create synced table for customer demographics
customer_demographics_table = f"{CATALOG}.{SCHEMA}.customer_demographics"

customer_demographics.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(customer_demographics_table)

print(f"✓ Created table: {customer_demographics_table}")


## 3. Time Series Sales Table

Daily aggregations for time series analysis.


In [ ]:
# Daily aggregations
time_series_daily = (
    transactions_df
    .groupBy("t_dat", "year", "month", "year_month")
    .agg(
        count("*").alias("num_transactions"),
        sum("price").alias("total_revenue"),
        countDistinct("customer_id").alias("unique_customers"),
        countDistinct("article_id").alias("unique_products"),
        avg("price").alias("avg_transaction_value")
    )
    .withColumn("date", col("t_dat"))
    .select(
        "date",
        "year",
        "month",
        "year_month",
        "num_transactions",
        "total_revenue",
        "unique_customers",
        "unique_products",
        "avg_transaction_value"
    )
    .orderBy("date")
)

print(f"Time series data: {time_series_daily.count():,} days")
display(time_series_daily.limit(20))


In [ ]:
# Create synced table for time series
time_series_table = f"{CATALOG}.{SCHEMA}.time_series_sales"

time_series_daily.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(time_series_table)

print(f"✓ Created table: {time_series_table}")


## Summary


In [ ]:
with mlflow.start_run(run_name="prepare_app_features") as run:
    
    # Log table names
    tables = [
        product_sales_table,
        category_insights_table,
        customer_demographics_table,
        time_series_table
    ]
    
    for table in tables:
        mlflow.log_param(f"table_{table.split('.')[-1]}", table)
    
    # Log row counts
    mlflow.log_metric("product_sales_rows", product_sales_summary.count())
    mlflow.log_metric("category_insights_rows", category_insights.count())
    mlflow.log_metric("customer_demographics_rows", customer_demographics.count())
    mlflow.log_metric("time_series_rows", time_series_daily.count())
    
    mlflow.set_tag("stage", "app_preparation")
    
    print("\n" + "="*60)
    print("APP FEATURES PREPARATION SUMMARY")
    print("="*60)
    print("\nCreated Delta tables:")
    for table in tables:
        count = spark.table(table).count()
        print(f"  ✓ {table}: {count:,} rows")
    print("\nThese Delta tables are ready to be synced to Lakebase.")
    print("="*60)
